In [1]:
import numpy as np


# Section 4?
def perlin_generate_permutation(seed = None):
	rng = np.random.default_rng(seed)
	p = np.arange(256, dtype = np.int32)
	rng.shuffle(p)
	return np.concatenate([p, p]).astype(np.int32)


# Deficiency correction in Perlin 2002
def deficiency_correction(t):
	#6t^5 - 15t^4 + 10t^3
	return t * t * t * (t * (t * 6 - 15) + 10)


def lerp(t, a, b):
	return a + t * (b - a)


def perlin_grad(hashval, x, y, z):
	"""
	Gradient selection function from Perlin's 2002 reference implementation.
	"""
	h = hashval & 15
	u = np.where(h < 8, x, y)
	v = np.where(h < 4, y, np.where((h == 12) | (h == 14), x, z))
	return np.where((h & 1) == 0, u, -u) + np.where((h & 2) == 0, v, -v)


def perlin_noise3(x, y, z, p):
	x = np.asarray(x, dtype = np.float64)
	y = np.asarray(y, dtype = np.float64)
	z = np.asarray(z, dtype = np.float64)
	x, y, z = np.broadcast_arrays(x, y, z)

	X0 = np.floor(x).astype(np.int32) & 255
	Y0 = np.floor(y).astype(np.int32) & 255
	Z0 = np.floor(z).astype(np.int32) & 255

	xf = x - np.floor(x)
	yf = y - np.floor(y)
	zf = z - np.floor(z)

	u = deficiency_correction(xf)
	v = deficiency_correction(yf)
	w = deficiency_correction(zf)

	A = p[X0] + Y0
	AA = p[A] + Z0
	AB = p[A + 1] + Z0
	B = p[X0 + 1] + Y0
	BA = p[B] + Z0
	BB = p[B + 1] + Z0

	return lerp(
		w,
		lerp(
			v,
			lerp(
				u,
				perlin_grad(p[AA], xf, yf, zf),
				perlin_grad(p[BA], xf - 1, yf, zf),
			),
			lerp(
				u,
				perlin_grad(p[AB], xf, yf - 1, zf),
				perlin_grad(p[BB], xf - 1, yf - 1, zf),
			),
		),
		lerp(
			v,
			lerp(
				u,
				perlin_grad(p[AA + 1], xf, yf, zf - 1),
				perlin_grad(p[BA + 1], xf - 1, yf, zf - 1),
			),
			lerp(
				u,
				perlin_grad(p[AB + 1], xf, yf - 1, zf - 1),
				perlin_grad(p[BB + 1], xf - 1, yf - 1, zf - 1),
			),
		),
	)


def perlin_noise2(x, y, p):
	"""
	2D improved Perlin noise as a z=0 slice of the 3D function.
	"""
	x = np.asarray(x, dtype=np.float64)
	y = np.asarray(y, dtype=np.float64)
	x, y = np.broadcast_arrays(x, y)
	z = np.zeros_like(x, dtype=np.float64)
	return perlin_noise3(x, y, z, p)


def perlin_grid2(shape, scale, p, offset=(0.0, 0.0)):
	"""
	Generate a 2D array of improved Perlin noise.

	Parameters
	----------
	shape : tuple[int, int]
		(height, width)
	scale : float
		Larger values produce larger, smoother features.
	p : ndarray, shape (512,)
		Duplicated permutation table.
	offset : tuple[float, float]
		Offset in noise space, as (oy, ox).

	Returns
	-------
	ndarray
		Noise image of shape `shape`.
	"""
	if scale <= 0:
		raise ValueError("scale must be > 0")

	h, w = shape
	oy, ox = offset

	ys = np.arange(h, dtype=np.float64) / scale + oy
	xs = np.arange(w, dtype=np.float64) / scale + ox
	yy, xx = np.meshgrid(ys, xs, indexing="ij")

	return perlin_noise2(xx, yy, p)

In [8]:
from PIL import Image

p = perlin_generate_permutation(seed=42)
img = perlin_grid2((2048, 2048), scale=32.0, p=p)

img01 = np.clip((img + 1.0) * 0.5, 0.0, 1.0)
img8 = (img01 * 255).astype(np.uint8)

Image.fromarray(img8, mode="L").save("perlin_2048scale32.png")